In [4]:
# Zelle zur Überprüfung des Arbeitsverzeichnisses
import os
print("Aktuelles Arbeitsverzeichnis (von hier aus sucht Python):")
print(os.getcwd())

Aktuelles Arbeitsverzeichnis (von hier aus sucht Python):
/Users/edinger-user/Desktop/Merten/HE_projekt/extern/MIDOGpp/test


In [2]:
# Zelle 1: Notwendige Imports

import pandas as pd
import sqlite3
import openslide
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import random

print("Module importiert. Bereit zum Start.")

Module importiert. Bereit zum Start.


In [6]:
# Zelle 2: Pfade und Konfiguration

# Pfade zu den Daten
image_folder = Path('../images/')
annotation_file = Path('../images/MIDOGpp.sqlite')
slide_info_file = Path('../datasets_xvalidation.csv') # Diese CSV enthält die Dateinamen

# Überprüfen, ob alles da ist
assert image_folder.exists(), f"FEHLER: Bild-Ordner '{image_folder}' nicht gefunden!"
assert annotation_file.exists(), f"FEHLER: Annotations-DB '{annotation_file}' nicht gefunden!"
assert slide_info_file.exists(), f"FEHLER: Slide-Info CSV '{slide_info_file}' nicht gefunden!"

print("Pfade sind korrekt.")

Pfade sind korrekt.


In [11]:
# Zelle 3: Zufälliges Bild auswählen

# Lade die Slide-Informationen
slides_df = pd.read_csv(slide_info_file, delimiter=";")
print(f"Insgesamt {len(slides_df)} Slides im Datensatz gefunden.")

# Wähle eine zufällige Slide aus der Liste
random_slide_info = slides_df.sample(1).iloc[0]
slide_uid = random_slide_info['Slide']
slide_tumortyp = random_slide_info['Tumor']

print(f"\nZufällige Slide ausgewählt:")
print(f"  -> UID: {slide_uid}")
print(f"  -> Tumortyp: {slide_tumortyp}")

Insgesamt 503 Slides im Datensatz gefunden.

Zufällige Slide ausgewählt:
  -> UID: 467
  -> Tumortyp: canine soft tissue sarcoma


In [12]:
# Zelle 4: Bildpfad und Annotationen laden

# Verbindung zur SQLite-Datenbank herstellen
conn = sqlite3.connect(annotation_file)
cursor = conn.cursor()

# 1. Finde den Dateinamen der Slide anhand der UID
cursor.execute("SELECT filename FROM Slides WHERE uid = ?", (slide_uid,))
result = cursor.fetchone()
if result is None:
    raise ValueError(f"Konnte keine Slide mit UID {slide_uid} in der Datenbank finden.")
slide_filename = result[0].replace('.png', '.tiff') # Die echten Dateien sind TIFFs
image_path = list(image_folder.glob(f"**/{slide_filename}"))[0]

print(f"Dateipfad gefunden: {image_path.name}")

# 2. Lade alle Annotationen für diese Slide
#    Wir nehmen nur die Klasse 1 (Mitotic Figure)
query = """
SELECT coord_x, coord_y, agreedClass
FROM Annotations_coordinates
JOIN Annotations ON Annotations_coordinates.annoId = Annotations.uid
WHERE Annotations.slide = ? AND Annotations.agreedClass = 1
"""
cursor.execute(query, (slide_uid,))
annotations = cursor.fetchall()
conn.close()

print(f"Insgesamt {len(annotations)} mitotische Figuren in dieser Slide gefunden.")

ValueError: Konnte keine Slide mit UID 467 in der Datenbank finden.

In [ ]:
# Zelle 5: Bild anzeigen und Annotationen einzeichnen

if not annotations:
    print("Keine Annotationen zum Anzeigen gefunden.")
else:
    # Öffne das WSI mit OpenSlide
    slide = openslide.OpenSlide(str(image_path))
    
    # Wir zentrieren den Ausschnitt um die erste gefundene Mitose
    # Die Koordinaten in der DB sind für Level 0 (höchste Auflösung)
    center_x, center_y, _ = annotations[0]
    
    # Definiere die Größe des Ausschnitts, den wir uns ansehen wollen
    patch_width = 1000
    patch_height = 1000
    
    # Berechne die obere linke Ecke des Ausschnitts
    location_x = int(center_x - patch_width / 2)
    location_y = int(center_y - patch_height / 2)
    
    # Lese den Bildausschnitt aus dem WSI
    patch = slide.read_region((location_x, location_y), 0, (patch_width, patch_height))
    
    # Schließe die Slide-Datei
    slide.close()

    # --- Jetzt zeichnen wir alles ---
    fig, ax = plt.subplots(1, figsize=(12, 12))
    ax.imshow(patch)
    ax.set_title(f"Ausschnitt aus Slide {slide_uid} ({slide_tumortyp})")

    # Gehe durch alle Annotationen und zeichne die, die im Patch sichtbar sind
    for x, y, _ in annotations:
        # Konvertiere absolute Koordinaten in relative Koordinaten des Patches
        relative_x = x - location_x
        relative_y = y - location_y

        # Prüfe, ob die Annotation im sichtbaren Bereich liegt
        if 0 <= relative_x < patch_width and 0 <= relative_y < patch_height:
            # Zeichne einen kleinen Kreis oder eine Box um die Mitose
            # Eine Mitose ist klein, ca. 30x30 Pixel
            rect = Rectangle((relative_x - 15, relative_y - 15), 30, 30, linewidth=2, edgecolor='lime', facecolor='none')
            ax.add_patch(rect)
            
    plt.show()